# Stage 4 — pdf_to_text

Re-create this stage's script with Gemini's help. The cells below give you the spec, the seed, the gotchas, and an inline eval. The implementation itself is yours to write.


## 1. Setup


Every cell in this section is idempotent and safe to re-run. If you opened this notebook fresh (without running anything else in the same runtime), run them top-to-bottom.


### Clone the repo and `cd` into it


In [ ]:
# Bootstrap: clone the workshop repo into /content and cd into it.
# Idempotent — safe to re-run.
import os, subprocess, sys
REPO_DIR = "/content/ar-bic-2026-workshop"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/jayprimer/ar-bic-2026-workshop.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


### Install dependencies


Python (`openai`) and the Node CLI `@llamaindex/liteparse`. First run takes ~30s; re-runs are near-instant.


In [ ]:
# Install dependencies. Idempotent (pip skips already-installed; npm re-link is cheap).
# liteparse only matters for Stage 4 but installing it everywhere keeps each
# notebook self-contained, which is the whole point of re-running this cell.
!pip install -q -r requirements.txt
!npm install -g @llamaindex/liteparse 2>&1 | tail -3


### (no API key needed for this stage)


In [ ]:
# This stage doesn't call the OpenAI API.


### Seed prior stages' outputs from the reference run


Stage 4 reads outputs from earlier stages. Each Colab notebook gets its own runtime, so work done in another notebook is not visible here. This cell seeds `stage_01..stage_03/data/` from the canonical reference run so Stage 4 has inputs to work with — but only when the dir is empty, so re-running an earlier stage IN THIS runtime is not clobbered.


In [ ]:
# Load prior stages' reference outputs as inputs for Stage 4.
# Each Colab notebook opens with a fresh runtime, so any work done in a
# Stage <4 notebook in a DIFFERENT runtime is not visible here.
# This cell makes the stage runnable in isolation against the canonical
# reference run. If you re-run an earlier stage IN THIS runtime, your
# output replaces these reference files (cwd is /content/...).
import os, shutil, glob
for n in range(1, 4):
    dst = f"stage_0{n}/data"
    src = f"reference_outputs/stage_0{n}/data"
    if not os.path.isdir(src):
        continue
    os.makedirs(dst, exist_ok=True)
    # Only seed if the participant hasn't produced anything for this stage
    # in the current runtime — otherwise we'd clobber their work.
    if any(os.scandir(dst)):
        print(f"skip stage_0{n} — already has files (keeping your work)")
        continue
    for src_file in glob.glob(f"{src}/*"):
        shutil.copy(src_file, dst)
    print(f"seeded stage_0{n}/data from reference_outputs")


## 2. Spec — paste this into Gemini


Open the Gemini side panel in Colab (sparkles icon, top right) and paste the block below as your prompt. Then iterate.

```
Convert every artifact under `stage_03/data/` to a `.md` under
`stage_04/data/`. Handle three input shapes uniformly:

  *.pdf  → run `lit parse <path> -o <tmp>.txt` and read the result
  *.xml  → walk JATS: emit `# <title>`, `## Abstract`, paragraphs, and
           heading-rendered <sec> structure. Drop figures, refs, math.
  *.json → render Stage 3 metadata fallback as title + abstract +
           authors/journal footer.

Both XML and metadata-fallback outputs should emit an authors/journal/
year/pub_types footer so the Stage 5 extractor sees a consistent shape.
```


## 3. Gotchas Gemini probably won't know


Copy any that apply into Gemini if it goes off-track:

- **liteparse is a Node CLI, not a Python lib.** Invoke via
  `subprocess.run(["lit", "parse", path, "-o", tmp_path], check=True)`.
  Installed in Section 1.
- **JATS uses `<sec>` and `<p>`.** Recurse with `element.iter()` or a
  depth-tracking walker — don't grab only top-level paragraphs.
- **Authors live in `<front>//<contrib-group>//<contrib>`** in JATS,
  with `<surname>` + `<given-names>` children. Bodies have no author
  byline — without a footer, Stage 5 hallucinates first_author.
- **`itertext()` again.** Same trick as Stage 1 for any element that
  might wrap inline children.
- **Floor each output.** Fulltext output should be much larger than an
  abstract — assert `getsize > 5_000` (or `> 500` for the JSON
  fallback) to catch parse failures early.


## 4. Seed — a few lines to anchor Gemini


In [ ]:
import glob, json, os, re, subprocess, tempfile
import xml.etree.ElementTree as ET
os.makedirs("stage_04/data", exist_ok=True)
# Read inputs from stage_03/data/*.{pdf,xml,json} and write
# stage_04/data/<stem>.md for each. The inspect cell will glob the
# results — no Python output variable required.


## 5. Your implementation


Drive Gemini to fill this in. Iterate until the inspect cell below shows reasonable output and the eval cell passes.


In [ ]:
# TODO: implement Stage 4 here.
# Read the spec above. Use the seed cell's imports.
# When done, run the inspect + eval cells next.


## 6. Inspect output


In [ ]:
import glob, os
mds = sorted(glob.glob("stage_04/data/*.md"))
print(f"{len(mds)} .md files in stage_04/data/\n")
for p in mds:
    kb = os.path.getsize(p) // 1024
    stem = os.path.splitext(os.path.basename(p))[0]
    with open(p) as f:
        first = f.readline().strip()[:78]
    print(f"  {stem:18s} {kb:>5d} KB   {first}")


## 7. Run eval


Inline eval — same checks as `eval/eval_04_script.py`, but the code is right here so you can see what it's measuring. Writes `stage_04/eval/eval_script.json` + `score.json`.


In [ ]:
# Same checks as eval/eval_04_script.py, inlined.
import json, os, re
os.makedirs("stage_04/eval", exist_ok=True)
with open("stage_03/data/fetched.json") as f:
    fetched = json.load(f)
with open("stage_02/data/screened.json") as f:
    abstract_by_pmid = {r["pmid"]: r["abstract"] for r in json.load(f)}

def first_words(s, n=80):
    return re.sub(r"\s+", " ", (s or "")).strip()[:n]
def alnum(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())
def _floor(path):
    return 500 if path.endswith(".json") else 5_000

errors, items = [], []
for pmid, path in fetched.items():
    if not path: continue
    stem = os.path.splitext(os.path.basename(path))[0]
    text_path = f"stage_04/data/{stem}.md"
    if not os.path.exists(text_path):
        errors.append(f"{pmid} ({stem}): text missing")
        items.append({"pmid": pmid, "stem": stem, "size_ok": False, "abstract_roundtrip": False})
        continue
    size_ok = os.path.getsize(text_path) > _floor(path)
    text = open(text_path).read()
    needle = first_words(abstract_by_pmid.get(pmid, ""))
    rt = bool(needle) and alnum(needle) in alnum(text)
    items.append({"pmid": pmid, "stem": stem, "size_ok": size_ok,
                  "abstract_roundtrip": rt, "needle": needle})
    if not size_ok: errors.append(f"{pmid} ({stem}): < size floor")
    if not rt and needle: errors.append(f"{pmid} ({stem}): abstract doesn't round-trip")

n_files = len(items)
n_size_ok = sum(1 for it in items if it["size_ok"])
n_rt = sum(1 for it in items if it["abstract_roundtrip"])
checks = {
    "every_screened_paper_has_text": all(
        os.path.exists(f"stage_04/data/{os.path.splitext(os.path.basename(p))[0]}.md")
        for p in fetched.values() if p),
    "all_files_above_size_floor":  n_size_ok == n_files,
    "all_abstracts_round_trip":    n_rt == n_files,
}
print("Script checks:")
for k, v in checks.items():
    print(f"  {'OK  ' if v else 'FAIL'}  {k}")
if errors:
    print("Errors:")
    for e in errors: print(f"    {e}")
print(f"\nRound-trip: {n_rt}/{n_files} abstract prefixes match converted text")

with open("stage_04/eval/eval_script.json", "w") as f:
    json.dump({"script": checks, "items": items}, f, indent=2)
n_pass = sum(1 for v in checks.values() if v); n_total = len(checks)
score_path = "stage_04/eval/score.json"
score = json.load(open(score_path)) if os.path.exists(score_path) else {}
score["script"] = {"passed": n_pass, "total": n_total,
                   "percent": round(100*n_pass/n_total, 1)}
with open(score_path, "w") as f: json.dump(score, f, indent=2)
print(f"\nScore: {n_pass}/{n_total} ({score['script']['percent']}%)")


## 8. Stuck? Skip this stage


Copy the reference run's Stage 4 output into place so the next stage's notebook can still run. Use this sparingly — the point of the workshop is to *re-create* each stage.


In [ ]:
import os, shutil, glob
os.makedirs("stage_04/data", exist_ok=True)
for src in glob.glob("reference_outputs/stage_04/data/*.md"):
    shutil.copy(src, "stage_04/data/")
print(f"copied {len(os.listdir('stage_04/data'))} reference .md files")
